# Mão na Roda — Fine-tuning do BERTimbau
Diagnóstico Preliminar de Falhas Automotivas Baseado em IA

Notebook único: setup → tokenização → treino ponderado (peso_treino) →
comparação das 3 proporções de split (80/10/10, 70/15/15, 60/20/20) →
treino completo na config vencedora → avaliação final no teste.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de
execução → GPU (T4).

## Passo 1 — Setup e verificação do ambiente

In [4]:
# --- 1.1 Instala as dependências ---
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl

# --- 1.2 Verifica se a GPU está disponível ---
import torch
print("Torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  Nenhuma GPU detectada. Ative em 'Ambiente de execução' antes de continuar.")

# --- 1.3 Constantes do projeto ---
MODEL_CHECKPOINT = "neuralmind/bert-base-portuguese-cased"  # BERTimbau (Souza et al., 2020)
SEED = 42
NUM_LABELS = 10
MAX_LENGTH = 64  # relatos são curtos (frases únicas); 64 tokens cobre a grande maioria

LABEL_NAMES = [
    "Bateria/Sistema Elétrico", "Sistema de Freios", "Superaquecimento do Motor",
    "Suspensão/Amortecedores", "Sistema de Transmissão/Câmbio", "Vazamento de Óleo",
    "Sistema de Arrefecimento", "Pneu/Roda", "Sistema de Injeção/Combustível",
    "Sistema de Escapamento",
]

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("\nSetup concluído. Modelo base:", MODEL_CHECKPOINT)

Torch: 2.11.0+cu128
GPU disponível: True
GPU: Tesla T4

Setup concluído. Modelo base: neuralmind/bert-base-portuguese-cased


In [5]:
# --- 1.4 Carregando os arquivos de split ---
from google.colab import files
uploaded = files.upload()  # selecionem os 9 CSVs (treino/val/teste x 3 configs)

import pandas as pd
CONFIGS = ["80_10_10", "70_15_15", "60_20_20"]

raw = {}
for tag in CONFIGS:
    raw[tag] = {
        "treino": pd.read_csv(f"split_treino_{tag}.csv"),
        "val":    pd.read_csv(f"split_val_{tag}.csv"),
        "teste":  pd.read_csv(f"split_teste_{tag}.csv"),
    }
    n_tr, n_val, n_te = len(raw[tag]["treino"]), len(raw[tag]["val"]), len(raw[tag]["teste"])
    print(f"{tag}: treino={n_tr} | val={n_val} | teste={n_te} | total={n_tr+n_val+n_te}")

assert "peso_treino" in raw["80_10_10"]["treino"].columns, \
    "Coluna peso_treino não encontrada — confira se subiu o CSV certo (do v15)."
print("\nColuna peso_treino presente. Seguindo para tokenização.")

Saving split_teste_60_20_20.csv to split_teste_60_20_20.csv
Saving split_teste_70_15_15.csv to split_teste_70_15_15.csv
Saving split_teste_80_10_10.csv to split_teste_80_10_10.csv
Saving split_treino_60_20_20.csv to split_treino_60_20_20.csv
Saving split_treino_70_15_15.csv to split_treino_70_15_15.csv
Saving split_treino_80_10_10.csv to split_treino_80_10_10.csv
Saving split_val_60_20_20.csv to split_val_60_20_20.csv
Saving split_val_70_15_15.csv to split_val_70_15_15.csv
Saving split_val_80_10_10.csv to split_val_80_10_10.csv
80_10_10: treino=2432 | val=305 | teste=304 | total=3041
70_15_15: treino=2171 | val=435 | teste=435 | total=3041
60_20_20: treino=1824 | val=608 | teste=609 | total=3041

Coluna peso_treino presente. Seguindo para tokenização.


## Passo 2 — Tokenização

Convertemos cada split em um `Dataset` do Hugging Face, já tokenizado.
Para o **treino**, mantemos a coluna `peso_treino` (vira `weight`, usada na
Etapa 3 pela loss ponderada). Para **validação/teste**, `weight=1.0` — a
ponderação só faz sentido durante o treino; a avaliação deve refletir o
desempenho real, sem correção de amostragem.

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_fn(batch):
    return tokenizer(batch["relato"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

def build_hf_dataset(df, is_train):
    d = df[["relato", "falha_label"]].copy()
    d["labels"] = d["falha_label"].astype(int)
    d["weight"] = df["peso_treino"].astype(float) if is_train else 1.0
    ds = Dataset.from_pandas(d[["relato", "labels", "weight"]], preserve_index=False)
    ds = ds.map(tokenize_fn, batched=True)
    ds = ds.remove_columns(["relato"])
    return ds

datasets = {}
for tag in CONFIGS:
    datasets[tag] = {
        "treino": build_hf_dataset(raw[tag]["treino"], is_train=True),
        "val":    build_hf_dataset(raw[tag]["val"],    is_train=False),
        "teste":  build_hf_dataset(raw[tag]["teste"],  is_train=False),
    }
    print(tag, "-> treino:", len(datasets[tag]["treino"]),
          "| val:", len(datasets[tag]["val"]),
          "| teste:", len(datasets[tag]["teste"]))

print("\nExemplo tokenizado:", datasets["80_10_10"]["treino"][0].keys())

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2432 [00:00<?, ? examples/s]

... (7 outputs left out)

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

60_20_20 -> treino: 1824 | val: 608 | teste: 609

Exemplo tokenizado: dict_keys(['labels', 'weight', 'input_ids', 'token_type_ids', 'attention_mask'])


## Passo 3 — Trainer com perda ponderada (`peso_treino`)

O `Trainer` padrão do Hugging Face **não aplica pesos por amostra** —
precisamos sobrescrever `compute_loss` pra multiplicar a perda de cada
exemplo pelo seu `weight` antes de tirar a média. Isso é o que efetivamente
corrige o viés de superamostragem das famílias de paráfrase (fundamentação:
Shimodaira, 2000 — ver documentação técnica da base de dados).

**Atenção ao `remove_unused_columns=False`** nos argumentos de treino mais
abaixo — sem isso, o `Trainer` descarta a coluna `weight` automaticamente
por achar que é "não usada" pelo modelo, e a ponderação silenciosamente
deixa de funcionar.

In [7]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(reduction="none")
        per_example_loss = loss_fct(logits, labels)
        loss = (per_example_loss * weights).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

from sklearn.metrics import f1_score, accuracy_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

print("WeightedTrainer e compute_metrics definidos.")

WeightedTrainer e compute_metrics definidos.


## Passo 4 — Treino leve comparativo (3 configs)

Treino curto (poucas épocas) nas 3 proporções, só para comparar o F1-macro
na validação de cada uma e escolher a configuração vencedora. **Não usa o
conjunto de teste** — o teste fica intocado até o treino final (Passo 6).

In [8]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, DataCollatorWithPadding

LIGHT_EPOCHS = 3
BATCH_SIZE = 16

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

resultados_leves = {}

for tag in CONFIGS:
    print(f"\n{'='*60}\nTreino leve — config {tag}\n{'='*60}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS
    )

    args = TrainingArguments(
        output_dir=f"./checkpoints_leve_{tag}",
        num_train_epochs=LIGHT_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        remove_unused_columns=False,   # crítico: preserva a coluna 'weight'
        report_to="none",
        seed=SEED,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets[tag]["treino"],
        eval_dataset=datasets[tag]["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    resultados_leves[tag] = metrics["eval_f1_macro"]
    print(f"\n{tag} -> F1-macro (validação): {metrics['eval_f1_macro']:.4f}")

    del model, trainer
    torch.cuda.empty_cache()


Treino leve — config 80_10_10


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

... (10 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.677210,1.175300,0.721294,0.725329
2,0.747487,0.722567,0.818602,0.819079
3,0.493813,0.667392,0.801754,0.802632


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.493813,0.667392,3,0.801754,0.802632



60_20_20 -> F1-macro (validação): 0.8018


## Passo 5 — Comparação e escolha da configuração vencedora

In [9]:
print("Resultado do treino leve (F1-macro na validação):\n")
for tag, f1 in sorted(resultados_leves.items(), key=lambda x: -x[1]):
    print(f"  {tag}: {f1:.4f}")

CONFIG_VENCEDORA = max(resultados_leves, key=resultados_leves.get)
print(f"\nConfiguração vencedora: {CONFIG_VENCEDORA}")
print("\n⚠️  Se a diferença entre as duas melhores for pequena (<0.01-0.02),")
print("considerem empate técnico — nesse caso, priorizem a config com mais")
print("dados de treino (80/10/10), que tende a ser mais robusta e estável.")

# Para forçar manualmente uma config específica (ex.: por causa do empate
# técnico acima), descomentem e ajustem a linha abaixo:

CONFIG_VENCEDORA = "80_10_10"
print(f"\n✅ Configuração vencedora (definida manualmente): {CONFIG_VENCEDORA}")


Resultado do treino leve (F1-macro na validação):

  70_15_15: 0.8788
  80_10_10: 0.8717
  60_20_20: 0.8018

Configuração vencedora: 70_15_15

⚠️  Se a diferença entre as duas melhores for pequena (<0.01-0.02),
considerem empate técnico — nesse caso, priorizem a config com mais
dados de treino (80/10/10), que tende a ser mais robusta e estável.

✅ Configuração vencedora (definida manualmente): 80_10_10


In [10]:
print(f"\n✅ Configuração vencedora (definida manualmente): {CONFIG_VENCEDORA}")


✅ Configuração vencedora (definida manualmente): 80_10_10


## Passo 6 — Treino completo na configuração vencedora

Agora sim, o treino "de verdade" — mais épocas, com *early stopping*
monitorando o F1-macro da validação, e avaliação final no **teste**
(usado uma única vez, aqui). Esse é o número que vai para o TCC.

In [11]:
from transformers import EarlyStoppingCallback

FULL_EPOCHS = 15
PATIENCE = 3

print(f"Treino completo na config: {CONFIG_VENCEDORA}")

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS
)

args_final = TrainingArguments(
    output_dir=f"./checkpoints_final_{CONFIG_VENCEDORA}",
    num_train_epochs=FULL_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer_final = WeightedTrainer(
    model=model_final,
    args=args_final,
    train_dataset=datasets[CONFIG_VENCEDORA]["treino"],
    eval_dataset=datasets[CONFIG_VENCEDORA]["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

trainer_final.train()

Treino completo na config: 80_10_10


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.136174,0.739540,0.833146,0.832787
2,0.419840,0.418470,0.869232,0.868852
3,0.200698,0.435055,0.871530,0.872131
4,0.155097,0.422150,0.888455,0.888525
5,0.089108,0.448666,0.894555,0.895082
6,0.032825,0.542633,0.897326,0.898361
7,0.020197,0.562329,0.897184,0.898361
8,0.015633,0.565630,0.892504,0.891803
9,0.007159,0.620742,0.891696,0.891803


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1368, training_loss=0.3121165677207952, metrics={'train_runtime': 457.4741, 'train_samples_per_second': 79.742, 'train_steps_per_second': 4.984, 'total_flos': 719923554975744.0, 'train_loss': 0.3121165677207952, 'epoch': 9.0})

In [12]:
# --- Avaliação final no conjunto de TESTE (uso único) ---
pred_output = trainer_final.predict(datasets[CONFIG_VENCEDORA]["teste"])
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

print(f"Resultado final — config {CONFIG_VENCEDORA} — conjunto de TESTE\n")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=3))

f1_final = f1_score(y_true, y_pred, average="macro")
acc_final = accuracy_score(y_true, y_pred)
print(f"F1-macro final: {f1_final:.4f}")
print(f"Acurácia final: {acc_final:.4f}")

# --- Salva o modelo treinado ---
trainer_final.save_model(f"./modelo_bertimbau_final_{CONFIG_VENCEDORA}")
tokenizer.save_pretrained(f"./modelo_bertimbau_final_{CONFIG_VENCEDORA}")
print(f"\nModelo salvo em ./modelo_bertimbau_final_{CONFIG_VENCEDORA}")

# Para baixar o modelo pro Drive ou pro computador de vocês:
# from google.colab import files
# !zip -r modelo_final.zip ./modelo_bertimbau_final_{CONFIG_VENCEDORA}
# files.download('modelo_final.zip')

Resultado final — config 80_10_10 — conjunto de TESTE

                                precision    recall  f1-score   support

      Bateria/Sistema Elétrico      0.923     0.774     0.842        31
             Sistema de Freios      0.935     0.935     0.935        31
     Superaquecimento do Motor      0.679     0.633     0.655        30
       Suspensão/Amortecedores      0.900     0.871     0.885        31
 Sistema de Transmissão/Câmbio      0.906     0.935     0.921        31
             Vazamento de Óleo      0.903     0.933     0.918        30
      Sistema de Arrefecimento      0.743     0.867     0.800        30
                     Pneu/Roda      0.933     0.933     0.933        30
Sistema de Injeção/Combustível      0.897     0.867     0.881        30
        Sistema de Escapamento      0.875     0.933     0.903        30

                      accuracy                          0.868       304
                     macro avg      0.869     0.868     0.867       304
       

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo salvo em ./modelo_bertimbau_final_80_10_10


In [13]:
from google.colab import files
!zip -r modelo_final.zip ./modelo_bertimbau_final_80_10_10
files.download('modelo_final.zip')

  adding: modelo_bertimbau_final_80_10_10/ (stored 0%)
  adding: modelo_bertimbau_final_80_10_10/training_args.bin (deflated 53%)
  adding: modelo_bertimbau_final_80_10_10/config.json (deflated 60%)
  adding: modelo_bertimbau_final_80_10_10/tokenizer.json (deflated 72%)
  adding: modelo_bertimbau_final_80_10_10/model.safetensors (deflated 7%)
  adding: modelo_bertimbau_final_80_10_10/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>